In [ ]:
from pyscf import gto, scf
from qarp.operators import JordanWigner
from qarp.operators.pyscf import fermion_operator_from_mf, onv_from_mf

# build a qubit hamiltonian - we'll choose H2.

mol = gto.M(atom="H 0 0 0; H 0 0 0.735", basis="sto3g")
mol.build()
mf = scf.RHF(mol)
mf.kernel()

fop = fermion_operator_from_mf(mf)
onv = onv_from_mf(mf)

qop = JordanWigner().encode_operator(fop)

In [ ]:
from qarp.blocks import UCCBlock

# create a wfn
wfn = UCCBlock(occupation_number_vector=onv, generalised=True)

In [ ]:
from qarp.algorithms import SSVQE
from qarp.engines import QarpEngine
from qarp.optimizers import ScipyOptimizer
from qarp.blocks import ComputationalBasisStateBlock

ssvqe = SSVQE(
    operator=qop,
    ansatz_block=wfn,
    basis_state_blocks=[
        ComputationalBasisStateBlock([0, 0, 0, 0]).build(),
        ComputationalBasisStateBlock([1, 0, 1, 0]).build(),
        ComputationalBasisStateBlock([0, 1, 0, 1]).build(),
        ComputationalBasisStateBlock([1, 0, 0, 1]).build(),
        ComputationalBasisStateBlock([0, 1, 1, 0]).build(),
        ComputationalBasisStateBlock([1, 1, 1, 1]).build()
    ],
    weights=[48, 24, 12, 6, 3, 1],
    verbose=True,
)
ssvqe.build()
ssvqe.run()

In [ ]:
from qarp.operators.functions import eigenspectrum

print("expvals found by ssvqe:", ssvqe.energies)
print("eigenspectrum:", eigenspectrum(qop))

In [ ]:
ssvqe = SSVQE(
    operator=qop,
    ansatz_block=wfn,
    basis_state_blocks=[
        ComputationalBasisStateBlock([0, 0, 0, 0]).build(),
        ComputationalBasisStateBlock([1, 0, 1, 0]).build(),
        ComputationalBasisStateBlock([0, 1, 0, 1]).build(),
        ComputationalBasisStateBlock([1, 0, 0, 1]).build(),
        ComputationalBasisStateBlock([0, 1, 1, 0]).build(),
        ComputationalBasisStateBlock([1, 1, 1, 1]).build()
    ],
    weights=[48, 24, 12, 6, 3, 1],
    # Gradient-free optimizer
    optimizer=ScipyOptimizer("COBYLA"),
    verbose=True,
)
ssvqe.build()
ssvqe.run()

In [ ]:
print("expvals found by ssvqe:", ssvqe.energies)